In [1]:
import pandas as pd
import numpy as np

In [2]:
general_df = pd.read_csv('./generalizability/ldaMataveMetrics.csv')

In [3]:
general_df

,dataset,model,CHI,ZIPF,CLASSIFIER,IRPR,FID,PR,DC,MAUVE,TRADITIONAL,ZERO
0,yahoo,LDA,9.436303e-01,0.122554,0.888889,0.412986,1.012399,0.349915,0.103568,0.713425,0.374836,0.270910
1,yahoo,MATAVE,9.916591e-01,0.222275,0.962963,0.416155,1.028472,0.376042,0.096035,0.772034,0.562229,0.295235
2,yahoo,combinedTopicModel,9.995372e-01,0.158400,0.874558,0.414935,0.991504,0.214404,0.104635,0.720164,0.404874,0.252905
3,yahoo,realToReal,1.000000e+00,0.005036,0.416854,0.403372,0.880048,0.010016,0.029556,0.039246,0.148319,0.100081
4,yahoo,realToReal2,1.000000e+00,0.007992,0.593496,0.401987,0.865898,0.008375,0.012696,0.049415,0.155089,0.101293
5,yahoo,realToReal3,1.000000e+00,0.010083,0.373745,0.401627,0.868195,0.011692,0.014183,0.039774,0.177254,0.101734
6,banking77,LDA,2.776298e-13,0.131400,0.903989,0.310814,0.615170,0.382805,0.308852,0.703925,0.365555,0.147635
7,banking77,MATAVE,6.055634e-03,0.176033,0.991870,0.337082,0.716313,0.469054,0.637870,0.882104,0.445697,0.220855
8,banking77,combinedTopicModel,9.495060e-01,0.165787,0.934553,0.322378,0.623337,0.340998,0.442302,0.660813,0.396567,0.186324
9,banking77,realToReal,1.000000e+00,0.018521,0.609782,0.258443,0.365246,0.038515,0.007754,0.028652,0.154659,0.084253


In [4]:
calibrated_dfs = []
for dataset in general_df['dataset'].value_counts().keys().tolist():
    temp_df = general_df[general_df['dataset'] == dataset]
    temp_df = temp_df.drop(columns='dataset')

    # Get metric columns. 
    metric_cols = [c for c in temp_df.columns if 'model' not in c]
    # Extract real rows.
    real_mask = temp_df['model'].str.startswith('real')
    real_df = temp_df.loc[real_mask, metric_cols]

    # There should be a minimum standard deviation, otherwise the metric is returning all 0s or exhibits no variation (which we assume is not indicative of a perfect metric, but a failing one).
    min_std = 1e-6
    # Get standard deviation. 
    std_dev = real_df.std(ddof=1)
    # Get mean. 
    mean = real_df.mean()

    # Remove saturated metrics
    valid_metrics = pd.Series(True, index=metric_cols)
    for m in metric_cols:
        # Real-vs-real values for this metric
        real_values = real_df[m]
        # Detect saturation at minimum or maximum or standard deviation is less than min_std
        if (np.allclose(real_values, 1, atol=min_std)
            or np.allclose(real_values, 0, atol=min_std)
            or std_dev[m] < min_std
        ):
            valid_metrics[m] = False

    # Get validity weights (as inverse score such that lower scores get higher weight) and remove saturated metrics. 
    validity_weights  = (1 - mean).where(valid_metrics, 0)
    # Normalize (relative importance, not arbitrary magnitude). 
    validity_weights = validity_weights / validity_weights.sum()

    # Get reliability weights (coefficient of variation, which is dimensionless and scale invariant).
    safe_std = std_dev.clip(lower=min_std)
    reliability_weights = 1 - (safe_std / mean)
    # Remove saturated metrics. 
    reliability_weights = reliability_weights.where(valid_metrics, 0)
    # Normalize (relative importance, not arbitrary magnitude). 
    reliability_weights = reliability_weights / reliability_weights.sum()

    # Combine weights. 
    weights = (validity_weights + reliability_weights) / 2
    # Normalize (relative importance, not arbitrary magnitude). 
    weights = weights / weights.sum()

    weight_rank = weights.rank(ascending=False, method="dense").astype(int)
    
    # Calibrate scores.
    calibrated = temp_df.copy()

    for m in metric_cols:
        calibrated[f'weighted_{m}'] = (
            calibrated[m] * weights[m]
        )
        calibrated[f"rank_{m}"] = weight_rank[m]
        calibrated[f"weight_{m}"] = weights[m]

    calibrated["final_score"] = (
        calibrated[metric_cols] * weights
    ).sum(axis=1)

    calibrated['dataset'] = dataset

    calibrated_dfs.append(calibrated)

In [5]:
weights

CHI            0.067231
ZIPF           0.099962
CLASSIFIER     0.094232
IRPR           0.119383
FID            0.116238
PR             0.120072
DC             0.051118
MAUVE          0.082113
TRADITIONAL    0.122481
ZERO           0.127171
dtype: float64

In [6]:
calibrated_results = pd.concat(calibrated_dfs)
calibrated_results

,model,CHI,ZIPF,CLASSIFIER,IRPR,FID,PR,DC,MAUVE,TRADITIONAL,...,rank_MAUVE,weight_MAUVE,weighted_TRADITIONAL,rank_TRADITIONAL,weight_TRADITIONAL,weighted_ZERO,rank_ZERO,weight_ZERO,final_score,dataset
0,LDA,9.436303e-01,0.122554,0.888889,0.412986,1.012399,0.349915,0.103568,0.713425,0.374836,...,3,0.126799,0.045331,4,0.120935,0.035471,1,0.130934,0.441099,yahoo
1,MATAVE,9.916591e-01,0.222275,0.962963,0.416155,1.028472,0.376042,0.096035,0.772034,0.562229,...,3,0.126799,0.067993,4,0.120935,0.038656,1,0.130934,0.496636,yahoo
2,combinedTopicModel,9.995372e-01,0.158400,0.874558,0.414935,0.991504,0.214404,0.104635,0.720164,0.404874,...,3,0.126799,0.048964,4,0.120935,0.033114,1,0.130934,0.427661,yahoo
3,realToReal,1.000000e+00,0.005036,0.416854,0.403372,0.880048,0.010016,0.029556,0.039246,0.148319,...,3,0.126799,0.017937,4,0.120935,0.013104,1,0.130934,0.188392,yahoo
4,realToReal2,1.000000e+00,0.007992,0.593496,0.401987,0.865898,0.008375,0.012696,0.049415,0.155089,...,3,0.126799,0.018756,4,0.120935,0.013263,1,0.130934,0.203481,yahoo
5,realToReal3,1.000000e+00,0.010083,0.373745,0.401627,0.868195,0.011692,0.014183,0.039774,0.177254,...,3,0.126799,0.021436,4,0.120935,0.013321,1,0.130934,0.186464,yahoo
6,LDA,2.776298e-13,0.131400,0.903989,0.310814,0.615170,0.382805,0.308852,0.703925,0.365555,...,3,0.112363,0.040700,4,0.111338,0.017261,1,0.116918,0.386532,banking77
7,MATAVE,6.055634e-03,0.176033,0.991870,0.337082,0.716313,0.469054,0.637870,0.882104,0.445697,...,3,0.112363,0.049623,4,0.111338,0.025822,1,0.116918,0.491546,banking77
8,combinedTopicModel,9.495060e-01,0.165787,0.934553,0.322378,0.623337,0.340998,0.442302,0.660813,0.396567,...,3,0.112363,0.044153,4,0.111338,0.021785,1,0.116918,0.462227,banking77
9,realToReal,1.000000e+00,0.018521,0.609782,0.258443,0.365246,0.038515,0.007754,0.028652,0.154659,...,3,0.112363,0.017219,4,0.111338,0.009851,1,0.116918,0.205477,banking77


In [7]:
calibrated_results.to_csv('./generalizability.csv')

In [8]:
ranked_dict_results = []
for dataset in calibrated_results['dataset'].unique():
    subset = calibrated_results[calibrated_results['dataset'] == dataset]
    # Extract not real rows.
    real_mask = subset['model'].str.startswith('real')
    subset = subset.loc[~real_mask]
    ranks = subset['final_score'].rank(
        ascending=True,
        method='dense'
    ).astype(int)
    ranked_dict_results.append(pd.DataFrame({
        'model': subset['model'].values,
        'dataset': dataset,
        'final_score': subset['final_score'].values,
        'rank': ranks.values
    }))

In [9]:
ranked_results_df = pd.concat(ranked_dict_results)
ranked_results_df = ranked_results_df[
    (ranked_results_df['dataset'] != 'syntheticCareHomeNurseNotes') & 
    (ranked_results_df['dataset'] != 'atis') & 
    (ranked_results_df['dataset'] != 'clinc150')
    ]

In [10]:
ranked_results_df[ranked_results_df['rank'] == 1]['model'].value_counts()

model
combinedTopicModel    5
LDA                   2
Name: count, dtype: int64